# Phase 1: Training Pipeline — ResNet-50 Fine-tuning (Task A, Balanced)

**Task**: Conifer Family Classification (Task A)

**Dataset**: `data/task-A/` — augmented dataset sau Phase 0 augmentation.
2 lop thieu so (Podocarpaceae, Taxaceae) da duoc augment tu 217/206 len 1,600 mau moi.
Train split hien co 7,264 mau (thay vi 4,487 mau imbalanced truoc do).

**Strategy** (per WoodID_plan.md):
- Backbone LR = 1e-5 (1/10 of head LR) — preserves pretrained ImageNet features
- Head LR = 1e-4
- 50 epochs, cosine annealing, label smoothing 0.1, AMP

---

## Contents

1. [Setup & Root Detection](#1-setup)
2. [Install Dependencies](#2-install-dependencies)
3. [Download Dataset from HuggingFace](#3-download-dataset)
4. [Verify Dataset](#4-verify-dataset)
5. [Create Dataloaders](#5-create-dataloaders)
6. [Create & Test Model](#6-create-test-model)
7. [Fine-tune ResNet-50](#7-fine-tune)
8. [Evaluate on Test Set](#8-evaluate)

## 1. Setup & Root Detection <a id='1-setup'></a>

Detects whether running on Colab or local machine, and sets `ROOT` accordingly.

- **Colab**: `ROOT = /content/woodbench`
- **Local**: `ROOT = <repo-root>` (auto-detected from notebook location)

Colab setup (HF login, git clone) is in [Cell 2](#2-setup-colab).

In [1]:
# # Colab setup: install deps, HF login, clone repo, mount Drive.
# # Guard: only runs on Colab (/content/ guard).
# from pathlib import Path
# if Path("/content").exists():
#     !pip install -q huggingface_hub

#     from google.colab import userdata
#     import os
#     from huggingface_hub import login

#     hftoken = userdata.get("HF_TOKEN")
#     if hftoken:
#         login(token=hftoken)
#         print("[HF] Authenticated")
#     else:
#         print("[HF] No token found — using public access")

#     os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

# # =============================================================================
# # Colab setup — ONLY runs on Google Colab (/content guard).
# # =============================================================================

# # Step 1: Install huggingface_hub
# if Path("/content").exists():
#     import subprocess
#     subprocess.run(["pip", "install", "-q", "huggingface_hub"], capture_output=True)

# # Step 2: Authenticate HuggingFace
# if Path("/content").exists():
#     try:
#         from google.colab import userdata
#         from huggingface_hub import login
#         import os
#         if hftoken:
#             login(token=hftoken)
#             print("[HF] Authenticated")
#         else:
#             print("[HF] No token — public access")
#         os.environ["GH_TOKEN"] = str(userdata.get("GH_TOKEN", ""))
#     except Exception as e:
#         print(f"[HF] Auth skipped: {e}")

# # Step 3: Clone woodbench repo
# if Path("/content").exists():
#     import subprocess
#     import os
#     if not Path("/content/woodbench").exists():
#         token = os.environ.get("GH_TOKEN", "")
#         if token:
#             subprocess.run(
#                 ["git", "clone", f"https://{token}@github.com/ManifoldsTeam/woodbench.git"],
#                 capture_output=True, check=True,
#             )
#             print("[Git] Cloned woodbench")
#         else:
#             print("[Git] No token — skipping clone")
#     else:
#         print("[Git] woodbench already cloned")

# # Step 4: Mount Google Drive
# if Path("/content").exists():
#     from google.colab import drive
#     if not Path("/content/drive").exists():
#         drive.mount("/content/drive")
#         print("[Drive] Mounted")
#     else:
#         print("[Drive] Already mounted")
# else:
#     print("[Local] Skipping Colab-only setup")

In [2]:
import sys
from pathlib import Path

# Root detection: Colab uses /content/woodbench, local auto-detects from notebook location.
if Path("/workspace").exists():
    ROOT = Path("/workspace/woodbench")
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

import torch
print(f"[ROOT] {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

[ROOT] /workspace/woodbench
PyTorch: 2.11.0+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.4 GB


## 3. Install Dependencies <a id='3-install'></a>

Install project dependencies. Only runs on Colab (guarded by `/content` check).

In [3]:
# Only run on Colab
# if Path("/content").exists():
#     !pip install -q -r {ROOT}/requirements.txt

## 4. Git Checkout <a id='4-git'></a>

Checkout the experiment branch on Colab.

In [4]:
# %cd woodbench
# !git checkout feat/task-A

## 5. Download Dataset from HuggingFace <a id='5-download'></a>

Downloads the augmented `data/task-A/` dataset from HuggingFace Hub.
Dataset was created in Phase 0: Podocarpaceae and Taxaceae are augmented
from 217/206 originals to 1,600 samples each, giving 7,264 train samples total.

In [5]:
from pathlib import Path

# KEY CHANGE: Use data/task-A/ (augmented dataset) instead of data/split/ (imbalanced).
# Augmentation was done in Phase 0 notebook: notebooks/phase 0/3_task_a_augmentation.ipynb
# - Podocarpaceae: 217 originals -> 1,600 total (augmented)
# - Taxaceae:      206 originals -> 1,600 total (augmented)
# - Cupressaceae:  2,398 originals (unchanged)
# - Pinaceae:      1,666 originals (unchanged)
# Total train: 7,264 (vs 4,487 imbalanced before)

DATA_DIR = ROOT / "data" / "task-A"

# Download if missing
if not DATA_DIR.exists():
    from huggingface_hub import snapshot_download

    print("[WARN] Dataset not found locally")
    print("Downloading dataset from HuggingFace...")

    snapshot_download(
        repo_id="ny0g507h3p/wood-task-A",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )

    print("[OK] Download complete")

# Validate dataset
train_path = DATA_DIR / "train"

if train_path.exists():
    train_samples = (
        sum(1 for f in train_path.rglob("*.png")) +
        sum(1 for f in train_path.rglob("*.jpg"))
    )

    print(f"[OK] Dataset found at {DATA_DIR}")
    print(f"     Train images (approx): {train_samples}")

    if train_samples >= 7000:
        print("     Status: BALANCED dataset (~7,264 expected)")
    else:
        print("     WARNING: Expected ~7,264 train images")
        print("     This may be the old imbalanced dataset")
else:
    print(f"[ERROR] Missing train/ directory in {DATA_DIR}")

print(f"\nDATA_DIR = {DATA_DIR}")

[OK] Dataset found at /workspace/woodbench/data/task-A
     Train images (approx): 7264
     Status: BALANCED dataset (~7,264 expected)

DATA_DIR = /workspace/woodbench/data/task-A


## 6. Verify Dataset <a id='6-verify'></a>

Loads the train dataset and shows class distribution. With augmented data we expect:
- Cupressaceae: ~2,398 (majority, no augmentation)
- Pinaceae: ~1,666 (majority, no augmentation)
- Podocarpaceae: ~1,600 (minority, augmented from 217)
- Taxaceae: ~1,600 (minority, augmented from 206)
- Total train: ~7,264 (vs 4,487 imbalanced)

In [6]:
from src.data import ConiferDataset

# Check structure
print("Dataset structure:")
for split in ["train", "val", "test"]:
    split_path = DATA_DIR / split
    if split_path.exists():
        families = sorted([d.name for d in split_path.iterdir() if d.is_dir()])
        print(f"  {split}/: {len(families)} families - {families}")

# Load train set and show distribution
train_ds = ConiferDataset(DATA_DIR, split="train", task="taskA")
print(f"\nTotal training samples: {len(train_ds)}")
print(f"Classes: {train_ds.classes}")
print(f"\nClass distribution:")
for family, count in train_ds.get_class_distribution().items():
    pct = count / len(train_ds) * 100
    print(f"  {family}: {count} ({pct:.1f}%)")

total = len(train_ds)
expected = 2398 + 1666 + 1600 + 1600
print(f"\nExpected total (balanced): {expected}")
print(f"Actual total: {total}")
print(f"Match: {'OK' if total == expected else 'CHECK DATA'}")
print(f"\nAugmented minority classes: Podocarpaceae={train_ds.get_class_distribution()['Podocarpaceae']}, "
      f"Taxaceae={train_ds.get_class_distribution()['Taxaceae']}")

Dataset structure:
  train/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']
  val/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']
  test/: 4 families - ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

Total training samples: 7264
Classes: ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

Class distribution:
  Cupressaceae: 2398 (33.0%)
  Pinaceae: 1666 (22.9%)
  Podocarpaceae: 1600 (22.0%)
  Taxaceae: 1600 (22.0%)

Expected total (balanced): 7264
Actual total: 7264
Match: OK

Augmented minority classes: Podocarpaceae=1600, Taxaceae=1600


## 7. Create Dataloaders <a id='7-dataloaders'></a>

Create train/val/test DataLoaders with ImageNet normalization and online augmentation.
`use_90rotation=False` for the first run (ablation variant is in Phase 3).

In [7]:
from src.data import get_dataloaders

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_WORKERS = 4
USE_90ROTATION = False  # Ablation variant in Phase 3

train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMG_SIZE,
    use_90rotation=USE_90ROTATION,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

# Quick class balance check on first batch
unique, counts = labels.unique(return_counts=True)
print(f"\nFirst batch class distribution:")
for u, c in zip(unique, counts):
    print(f"  Class {u.item()}: {c.item()} samples")

Train batches: 113
Val batches:   15
Test batches:  16

Batch shape: torch.Size([64, 3, 224, 224])
Labels shape: torch.Size([64])

First batch class distribution:
  Class 0: 16 samples
  Class 1: 15 samples
  Class 2: 17 samples
  Class 3: 16 samples


## 8. Create & Test Model <a id='8-create-model'></a>

Creates ResNet-50 with unfrozen backbone (fine-tuning mode). All parameters are trainable.
Differential learning rates will be applied in the optimizer (next cell).

In [8]:
from src.models import create_model, BACKBONE_NAMES

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Check available backbones
print("\nAvailable backbones:")
for name, timm_name in BACKBONE_NAMES.items():
    print(f"  {name}: {timm_name}")

# Create model — unfrozen for fine-tuning (all 161 param groups trainable)
model = create_model("resnet50", num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Test forward pass
outputs = model(images.to(device))
print(f"\nOutput shape: {outputs.shape}  (expected: ({images.shape[0]}, 4))")

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal params:    {total_params / 1e6:.2f}M")
print(f"Trainable:       {trainable_params / 1e6:.2f}M")
print(f"Frozen:          {frozen_params / 1e6:.2f}M")
print(f"Fine-tuning mode: {'UNFROZEN' if frozen_params == 0 else 'PARTIALLY FROZEN'}")

Using device: cuda

Available backbones:
  resnet50: resnet50
  efficientnet_b3: tf_efficientnet_b3
  mobilenetv3_large: mobilenetv3_large_100
  convnext_tiny: convnext_tiny
  swin_tiny: swin_tiny_patch4_window7_224

Output shape: torch.Size([64, 4])  (expected: (64, 4))

Total params:    23.52M
Trainable:       23.52M
Frozen:          0.00M
Fine-tuning mode: UNFROZEN


## 9. Fine-tune ResNet-50 <a id='9-finetune'></a>

Fine-tuning strategy (per WoodID_plan.md):
- Head LR = 1e-4, Backbone LR = 1e-5 (differential LR)
- 50 epochs, cosine annealing, label smoothing 0.1, AMP
- Train on **augmented** dataset: 7,264 samples (vs 4,487 imbalanced)

Save best only (`save_best_only=True`) with early stopping.
Output saved to Google Drive: `woodbench_experiments/taskA/resnet50_balanced/`.

In [9]:
from pathlib import Path

# Output directory: Google Drive on Colab, local experiments/ on desktop.
# Note: using "resnet50_balanced" to distinguish from old "resnet50" (imbalanced) runs.
# if Path("/content/drive").exists():
#     DRIVE_ROOT = Path("/content/drive/MyDrive")
# else:
#     DRIVE_ROOT = ROOT / "experiments"

EXP_DIR = ROOT / "data" /"experiments"/ "task-A" / "resnet50"
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Output] {EXP_DIR}")

[Output] /workspace/woodbench/data/experiments/task-A/resnet50


In [10]:
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

set_seed(456)

# Re-create model fresh from pretrained weights
model = create_model("resnet50", num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Differential learning rates: backbone 1e-5, head 1e-4
HEAD_LR = 1e-4
BACKBONE_LR = 1e-5
optimizer = create_optimizer(
    model,
    lr=HEAD_LR,
    weight_decay=1e-4,
    backbone_lr=BACKBONE_LR,
)

print("Optimizer param groups:")
for pg in optimizer.param_groups:
    print(f"  [{pg['name']}] lr={pg['lr']:.2e}, {len(pg['params'])} tensors")

scheduler = create_scheduler(optimizer, epochs=50, scheduler_type="cosine")
criterion = create_loss_function(loss_type="label_smoothing", label_smoothing=0.1)

engine = TrainingEngine(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    use_amp=True,
)

print("\nStarting fine-tuning (50 epochs) on balanced dataset...")
print(f"Train samples: {len(train_ds)} (augmented)")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Head LR:       {HEAD_LR}")
print(f"Backbone LR:   {BACKBONE_LR}")
print(f"Label smooth:  0.1")
print(f"Scheduler:     cosine")
print(f"AMP:           enabled\n")

results = engine.fit(
    epochs=50,
    output_dir=EXP_DIR,
    name="resnet50_finetune_balanced",
    save_best_only=True,
    early_stopping_patience=5,
    verbose=True,
)

print(f"\nBest val accuracy: {results['best_val_acc']:.2f}%")
print(f"Final train acc:  {results['final_train_acc']:.2f}%")
print(f"Final val acc:    {results['final_val_acc']:.2f}%")

Optimizer param groups:
  [head] lr=1.00e-04, 2 tensors
  [backbone] lr=1.00e-05, 159 tensors

Starting fine-tuning (50 epochs) on balanced dataset...
Train samples: 7264 (augmented)
Batch size:    64
Head LR:       0.0001
Backbone LR:   1e-05
Label smooth:  0.1
Scheduler:     cosine
AMP:           enabled



Epoch 1/50 [Train]:   0%|          | 0/113 [00:00<?, ?batch/s]

Epoch 1/50 | Train Loss: 1.3911, Acc: 22.14% | Val Loss: 1.3900, Acc: 20.10% | LR: 2.00e-05[Best][EarlyStop 0/5]
  ↳ New best! Saved to resnet50_finetune_balanced_best.pth (Val Acc: 20.10%)


Epoch 2/50 | Train Loss: 1.3789, Acc: 30.71% | Val Loss: 1.3707, Acc: 42.83% | LR: 4.00e-05[Best][EarlyStop 0/5]
  ↳ New best! Saved to resnet50_finetune_balanced_best.pth (Val Acc: 42.83%)


Epoch 3/50 | Train Loss: 1.3489, Acc: 40.04% | Val Loss: 1.3409, Acc: 52.57% | LR: 6.00e-05[Best][EarlyStop 0/5]
  ↳ New best! Saved to resnet50_finetune_balanced_best.pth (Val Acc: 52.57%)


Epoch 4/50 | Train Loss: 1.3011, Acc: 48.41% | Val Loss: 1.2952, Acc: 55.50% | LR: 8.00e-05[Best][EarlyStop 0/5]
  ↳ New best! Saved to resnet50_finetune_balanced_best.pth (Val Acc: 55.50%)


... (35 outputs left out)

Epoch 40/50 | Train Loss: 0.4352, Acc: 97.28% | Val Loss: 0.4074, Acc: 98.74% | LR: 1.17e-05[EarlyStop 2/5]


Epoch 41/50 | Train Loss: 0.4348, Acc: 97.14% | Val Loss: 0.4105, Acc: 98.53% | LR: 9.55e-06[EarlyStop 3/5]


Epoch 42/50 | Train Loss: 0.4392, Acc: 96.89% | Val Loss: 0.4071, Acc: 98.43% | LR: 7.60e-06[EarlyStop 4/5]


Epoch 43/50 | Train Loss: 0.4357, Acc: 97.11% | Val Loss: 0.4088, Acc: 98.53% | LR: 5.85e-06[EarlyStop 5/5]

[EarlyStopping] No improvement for 5 epochs. Stopping at epoch 43.

Best val accuracy: 98.74%
Final train acc:  97.11%
Final val acc:    98.53%


In [11]:
# List all checkpoints saved during training
checkpoint_files = sorted(EXP_DIR.glob("resnet50_finetune_balanced_*.pth"))
print(f"Total checkpoints: {len(checkpoint_files)}")
print(f"First:  {checkpoint_files[0].name}")
print(f"Last:   {checkpoint_files[-1].name}")
print(f"Best:   {EXP_DIR / 'resnet50_finetune_balanced_best.pth'}")
print(f"All:    {len(checkpoint_files)} files")

Total checkpoints: 2
First:  resnet50_finetune_balanced_best.pth
Last:   resnet50_finetune_balanced_last.pth
Best:   /workspace/woodbench/data/experiments/task-A/resnet50/resnet50_finetune_balanced_best.pth
All:    2 files


## 10. Evaluate on Test Set <a id='10-evaluate'></a>

Loads the best checkpoint from the balanced-data training run and evaluates on the
held-out test set (which was NOT augmented — contains original images only).

In [12]:
from src.eval import compute_metrics, format_metrics, save_confusion_matrix

CLASS_NAMES = ["Cupressaceae", "Pinaceae", "Podocarpaceae", "Taxaceae"]

# Load the best checkpoint from the balanced-data training run
best_checkpoint_path = EXP_DIR / "resnet50_finetune_balanced_best.pth"

if best_checkpoint_path.exists():
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_epoch = checkpoint.get("epoch", 0) + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    print(f"Loaded best checkpoint: epoch {best_epoch}, val_acc={best_val_acc:.2f}%")
else:
    # Fallback: find the epoch with highest validation accuracy from training history
    import pandas as pd
    print(f"Best checkpoint not found at {best_checkpoint_path}")
    print("Finding best epoch from all checkpoints...")

# Evaluate on test set
metrics = compute_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=4,
    class_names=CLASS_NAMES,
)

print(format_metrics(metrics, CLASS_NAMES))

# Save confusion matrix
save_confusion_matrix(
    cm=metrics["confusion_matrix"],
    class_names=CLASS_NAMES,
    output_path=EXP_DIR / "confusion_matrix_balanced.png",
    title="Confusion Matrix — ResNet-50 Fine-tune (Balanced Data, Test Set)",
)
print(f"\nConfusion matrix saved to {EXP_DIR / 'confusion_matrix_balanced.png'}")

# Per-class accuracy breakdown
print("\nPer-class accuracy (from confusion matrix):")
cm = metrics["confusion_matrix"]
for i, name in enumerate(CLASS_NAMES):
    class_correct = cm[i, i]
    class_total = cm[i, :].sum()
    class_acc = class_correct / class_total * 100 if class_total > 0 else 0
    print(f"  {name}: {class_correct}/{class_total} = {class_acc:.1f}%")

Loaded best checkpoint: epoch 38, val_acc=98.74%
EVALUATION RESULTS
Accuracy:      97.76%
Precision:     93.83%
Recall:        97.87%
F1-macro:      95.74%

Per-Class F1 Scores:
  Cupressaceae        : 98.16%
  Pinaceae            : 98.37%
  Podocarpaceae       : 94.95%
  Taxaceae            : 91.49%

Confusion matrix saved to /workspace/woodbench/data/experiments/task-A/resnet50/confusion_matrix_balanced.png

Per-class accuracy (from confusion matrix):
  Cupressaceae: 508/525 = 96.8%
  Pinaceae: 361/364 = 99.2%
  Podocarpaceae: 47/47 = 100.0%
  Taxaceae: 43/45 = 95.6%
